In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
!pip install -U transformers datasets evaluate accelerate pyarrow scikit-learn

In [17]:
import os

data_dir = "/content/drive/MyDrive/SemEval2026/sem-eval-2026-task13-subtask-a"

print("Danh sách file:")
for f in os.listdir(data_dir):
    print("-", f)

Danh sách file:
- dataset_dict.json
- train
- validation


In [18]:
new_train_code = r'''
import os
import json
import argparse
import numpy as np
from datasets import load_from_disk
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
import evaluate


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_dir", type=str, required=True)
    parser.add_argument("--output_dir", type=str, required=True)
    parser.add_argument("--model_name", type=str, default="microsoft/unixcoder-base")
    parser.add_argument("--max_train_samples", type=int, default=None)
    parser.add_argument("--max_eval_samples", type=int, default=None)
    parser.add_argument("--num_train_epochs", type=int, default=3)
    parser.add_argument("--per_device_train_batch_size", type=int, default=4)
    parser.add_argument("--per_device_eval_batch_size", type=int, default=4)
    parser.add_argument("--gradient_accumulation_steps", type=int, default=4)
    parser.add_argument("--learning_rate", type=float, default=2e-5)
    parser.add_argument("--weight_decay", type=float, default=0.01)
    parser.add_argument("--logging_steps", type=int, default=3000)
    parser.add_argument("--save_total_limit", type=int, default=2)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--fp16", action="store_true")
    parser.add_argument("--bf16", action="store_true")
    parser.add_argument("--metric_for_best_model", type=str, default="f1")
    return parser.parse_args()


def detect_label_column(dataset):
    candidates = ["labels", "label", "target", "class", "y"]
    for col in candidates:
        if col in dataset.column_names:
            return col
    raise ValueError(f"Không tìm thấy cột nhãn. Các cột hiện có: {dataset.column_names}")


def detect_input_columns(dataset):
    has_ids = "input_ids" in dataset.column_names
    has_mask = "attention_mask" in dataset.column_names
    has_token_type_ids = "token_type_ids" in dataset.column_names

    if not has_ids:
        raise ValueError(f"Không tìm thấy cột input_ids. Các cột hiện có: {dataset.column_names}")

    return has_mask, has_token_type_ids


def infer_label_info(train_ds, label_col):
    values = train_ds[label_col]
    unique_labels = sorted(list(set(values)))

    if all(isinstance(x, (int, np.integer)) for x in unique_labels):
        label2id = {str(int(v)): int(v) for v in unique_labels}
        id2label = {int(v): str(int(v)) for v in unique_labels}
        num_labels = len(unique_labels)
    else:
        unique_labels = sorted([str(v) for v in unique_labels])
        label2id = {v: i for i, v in enumerate(unique_labels)}
        id2label = {i: v for v, i in label2id.items()}
        num_labels = len(unique_labels)

    return num_labels, label2id, id2label


def encode_labels(example, label_col, label2id):
    value = example[label_col]
    if isinstance(value, (int, np.integer)) and str(int(value)) in label2id:
        example["labels"] = int(label2id[str(int(value))])
    elif str(value) in label2id:
        example["labels"] = int(label2id[str(value)])
    else:
        raise ValueError(f"Nhãn không hợp lệ: {value}")
    return example


class TokenizedDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        import torch

        labels = [f["labels"] for f in features]
        features_no_labels = [{k: v for k, v in f.items() if k != "labels"} for f in features]

        batch = self.tokenizer.pad(
            features_no_labels,
            padding=True,
            return_tensors="pt"
        )

        batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch


def build_compute_metrics():
    accuracy_metric = evaluate.load("accuracy")
    precision_metric = evaluate.load("precision")
    recall_metric = evaluate.load("recall")
    f1_metric = evaluate.load("f1")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        accuracy = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
        precision = precision_metric.compute(predictions=preds, references=labels, average="macro")["precision"]
        recall = recall_metric.compute(predictions=preds, references=labels, average="macro")["recall"]
        f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]

        return {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }

    return compute_metrics


def main():
    args = parse_args()
    os.makedirs(args.output_dir, exist_ok=True)

    print("Đang load dataset từ:", args.data_dir)
    dataset_dict = load_from_disk(args.data_dir)

    label_col = detect_label_column(dataset_dict["train"])
    num_labels, label2id, id2label = infer_label_info(dataset_dict["train"], label_col)

    processed = {}
    for split in dataset_dict.keys():
        ds = dataset_dict[split].map(
            lambda x: encode_labels(x, label_col, label2id),
            desc=f"Encoding labels for {split}",
        )

        keep_cols = ["input_ids", "attention_mask", "labels"]
        remove_cols = [c for c in ds.column_names if c not in keep_cols]
        ds = ds.remove_columns(remove_cols)
        processed[split] = ds

    train_dataset = processed["train"]
    eval_dataset = processed["validation"]

    tokenizer = AutoTokenizer.from_pretrained(args.model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    config = AutoConfig.from_pretrained(
        args.model_name,
        num_labels=num_labels,
        label2id=label2id,
        id2label=id2label,
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        args.model_name,
        config=config,
        ignore_mismatched_sizes=True,
    )

    training_args = TrainingArguments(
        output_dir=args.output_dir,
        num_train_epochs=args.num_train_epochs,
        per_device_train_batch_size=args.per_device_train_batch_size,
        per_device_eval_batch_size=args.per_device_eval_batch_size,
        gradient_accumulation_steps=args.gradient_accumulation_steps,
        learning_rate=args.learning_rate,
        weight_decay=args.weight_decay,
        logging_steps=args.logging_steps,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=args.save_total_limit,
        save_steps=31250,
        load_best_model_at_end=True,
        metric_for_best_model=args.metric_for_best_model,
        greater_is_better=True,
        fp16=args.fp16,
        bf16=args.bf16,
        seed=args.seed,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=TokenizedDataCollator(tokenizer),
        compute_metrics=build_compute_metrics(),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()

    best_model_dir = os.path.join(args.output_dir, "best_model")
    os.makedirs(best_model_dir, exist_ok=True)

    trainer.save_model(best_model_dir)
    tokenizer.save_pretrained(best_model_dir)

    with open(os.path.join(best_model_dir, "label2id.json"), "w", encoding="utf-8") as f:
        json.dump(label2id, f, ensure_ascii=False, indent=2)

    with open(os.path.join(best_model_dir, "id2label.json"), "w", encoding="utf-8") as f:
        json.dump({str(k): v for k, v in id2label.items()}, f, ensure_ascii=False, indent=2)

    eval_metrics = trainer.evaluate(eval_dataset=eval_dataset)
    print("Validation metrics:", eval_metrics)

    meta = {
        "model_name": args.model_name,
        "task_name": os.path.basename(args.data_dir),
        "num_labels": num_labels,
        "label_column": label_col,
        "train_size": len(train_dataset),
        "validation_size": len(eval_dataset),
        "best_metric_name": args.metric_for_best_model,
        "validation_metrics": eval_metrics,
    }

    with open(os.path.join(best_model_dir, "model_meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    print("Da luu model tai:", best_model_dir)


if __name__ == "__main__":
    main()
'''
with open("train_unixcoder.py", "w", encoding="utf-8") as f:
    f.write(new_train_code)

print("Đã cập nhật train_unixcoder.py")

Đã cập nhật train_unixcoder.py


In [19]:
!python train_unixcoder.py \
  --data_dir "/content/drive/MyDrive/SemEval2026/sem-eval-2026-task13-subtask-a" \
  --output_dir "/content/drive/MyDrive/SemEval2026/outputs/unixcoder_subtask_a" \
  --num_train_epochs 3 \
  --per_device_train_batch_size 4 \
  --per_device_eval_batch_size 4 \
  --gradient_accumulation_steps 4 \
  --learning_rate 2e-5 \
  --weight_decay 0.01 \
  --logging_steps 3000 \
  --fp16

Đang load dataset từ: /content/drive/MyDrive/SemEval2026/sem-eval-2026-task13-subtask-a
Loading weights: 100% 197/197 [00:00<00:00, 84082.41it/s]
RobertaForSequenceClassification LOAD REPORT from: microsoft/unixcoder-base
Key                        | Status     | 
---------------------------+------------+-
embeddings.position_ids    | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
{'loss': '0.3122', 'grad_norm': '0.01309', 'learning_rate': '1.936e-05', 'epoch': '0.096'}
{'loss': '0.1839', 'grad_norm': '0.314', 'learning_r

In [20]:
output_dir = "/content/drive/MyDrive/SemEval2026/outputs/unixcoder_subtask_a/best_model"

print("Các file trong best_model:")
for f in os.listdir(output_dir):
    print("-", f)

Các file trong best_model:
- config.json
- model.safetensors
- tokenizer_config.json
- tokenizer.json
- training_args.bin
- label2id.json
- id2label.json
- model_meta.json


In [22]:
from transformers import AutoTokenizer

model_name = "microsoft/unixcoder-base"
best_model_dir = "/content/drive/MyDrive/SemEval2026/outputs/unixcoder_subtask_a/best_model"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.save_pretrained(best_model_dir)

print("Đã lưu lại tokenizer đầy đủ")
print(os.listdir(best_model_dir))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Đã lưu lại tokenizer đầy đủ
['config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin', 'label2id.json', 'id2label.json', 'model_meta.json']


In [23]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_dir = "/content/drive/MyDrive/SemEval2026/outputs/unixcoder_subtask_a/best_model"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

with open(f"{model_dir}/id2label.json", "r", encoding="utf-8") as f:
    id2label = {int(k): v for k, v in json.load(f).items()}

code_text = """
def add(a, b):
    return a + b
"""

inputs = tokenizer(
    code_text,
    truncation=True,
    padding=True,
    return_tensors="pt"
)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)[0]
    pred_id = torch.argmax(probs).item()

print({
    "predicted_label_id": pred_id,
    "predicted_label": id2label[pred_id],
    "confidence": float(probs[pred_id])
})

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'predicted_label_id': 0, 'predicted_label': '0', 'confidence': 0.9999959468841553}


In [21]:

  #--resume_checkpoint "/content/drive/MyDrive/SemEval2026/outputs/unixcoder_subtask_a/checkpoint-36000" \